<a href="https://colab.research.google.com/github/eshaans1305/AI-LAB/blob/main/lab2/eight_puzzle_IDS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
class PuzzleNode:
    def __init__(self, state, parent=None, action=None):
        self.state = state
        self.parent = parent
        self.action = action

    def __eq__(self, other):
        return self.state == other.state

    def __hash__(self):
        return hash(str(self.state))


def print_state(state):
    for row in state:
        print(" ".join(map(str, row)))
    print()


def actions(state):
    possible_actions = []
    empty_row, empty_col = find_empty(state)
    if empty_row > 0:
        possible_actions.append('up')
    if empty_row < 2:
        possible_actions.append('down')
    if empty_col > 0:
        possible_actions.append('left')
    if empty_col < 2:
        possible_actions.append('right')
    return possible_actions


def find_empty(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return i, j


def apply_action(state, action):
    new_state = [row[:] for row in state]
    empty_row, empty_col = find_empty(new_state)
    if action == 'up':
        new_state[empty_row][empty_col], new_state[empty_row - 1][empty_col] = \
        new_state[empty_row - 1][empty_col], new_state[empty_row][empty_col]
    elif action == 'down':
        new_state[empty_row][empty_col], new_state[empty_row + 1][empty_col] = \
        new_state[empty_row + 1][empty_col], new_state[empty_row][empty_col]
    elif action == 'left':
        new_state[empty_row][empty_col], new_state[empty_row][empty_col - 1] = \
        new_state[empty_row][empty_col - 1], new_state[empty_row][empty_col]
    elif action == 'right':
        new_state[empty_row][empty_col], new_state[empty_row][empty_col + 1] = \
        new_state[empty_row][empty_col + 1], new_state[empty_row][empty_col]
    return new_state


def depth_limited_search(initial_state, goal_state, limit):
    """
    Performs a Depth-Limited Search down to a maximum specified depth limit.
    Uses a path-checking set (local tracking) to avoid cyclical loops on the current branch.
    """
    # Stack stores tuples of (node, current_depth)
    stack = [(PuzzleNode(initial_state), 0)]

    # Track nodes currently on the active search path to avoid infinite loops
    path_set = set()

    while stack:
        current_node, depth = stack.pop()

        if current_node.state == goal_state:
            path = []
            node = current_node
            while node.parent is not None:
                path.append(node.action)
                node = node.parent
            path.reverse()
            return path

        # Only expand if we haven't hit the depth limit
        if depth < limit:
            # We track the path relative to the depth level to safely pop it later if needed,
            # but for standard iterative deepening on graphs, a simple path-set check suffices.
            path_set.add(current_node)

            for action in actions(current_node.state):
                next_state = apply_action(current_node.state, action)
                child_node = PuzzleNode(next_state, parent=current_node, action=action)

                # Avoid cycles along the current branch
                if child_node not in path_set:
                    stack.append((child_node, depth + 1))

    return None


def ids(initial_state, goal_state):
    """
    Iterative Deepening Search: incrementally increases the depth limit of DLS.
    """
    depth = 0
    while True:
        print(f"Searching at depth limit: {depth}...")
        result = depth_limited_search(initial_state, goal_state, depth)

        if result is not None:
            print(f"\nSolution found at depth {depth} in {len(result)} moves!")
            print("Moves:", " -> ".join(result))
            return result

        depth += 1


if __name__ == "__main__":
    start = [[2,8,3],
             [1,6,4],
             [7,0,5]
    ]

    goal = [[1,2,3],
            [8,0,4],
            [7,6,5]
    ]

    print("Initial State:")
    print_state(start)

    print("Goal State:")
    print_state(goal)

    print("Starting Iterative Deepening Search...")
    ids(start, goal)


Initial State:
2 8 3
1 6 4
7 0 5

Goal State:
1 2 3
8 0 4
7 6 5

Starting Iterative Deepening Search...
Searching at depth limit: 0...
Searching at depth limit: 1...
Searching at depth limit: 2...
Searching at depth limit: 3...
Searching at depth limit: 4...
Searching at depth limit: 5...

Solution found at depth 5 in 5 moves!
Moves: up -> up -> left -> down -> right
